# R3. Budget cases (30 min, teams of 2 to 3)
Each team draws one of **three cases**. The order matters:

1. **Look first.** See real inputs, the model's replies, and the **tokens and cost** of single items.
2. **Then predict** the bill for the full job, anchored on what you saw.
3. **Run a sample**, compare with your prediction, and **pull levers** to fit the budget.

**How the arena works**
- Your prompt can use placeholders: `{text}` (the item), `{labels}`, `{codebook}` (the official definitions), `{schema}` (extraction).
- The harness adds one output rule so everyone is parsed the same way. You do **not** need to ask for a specific format.
- Every run costs real money from your allowance and appears on the board. **Write your hypothesis in `note=` before you run.**
- The hidden test set stays with the instructor. Scores you see here are on the *dev* set.

In [ ]:
# ===== Run this cell first =====
import os, sys, hashlib, json
KIT_URL = "https://github.com/atsizelti/prompt_engineering.git"   # public repo with the kit
KIT_DIR = "arena_kit/prompt_arena_kit"                            # folder inside the clone
import importlib.util
if os.path.exists("arena_kit"):
    os.system("git -C arena_kit pull -q")                # always refresh a copy cloned earlier in this session
elif importlib.util.find_spec("arena") is None:
    os.system(f"git clone -q {KIT_URL} arena_kit")
if os.path.exists(KIT_DIR) and KIT_DIR not in sys.path:
    sys.path.insert(0, KIT_DIR)
import arena

NICK      = "your-nickname"      # pick a nickname (no real name needed)
PROVIDER  = "openai"             # "openai", or "mock" (practice with no key)
ENDPOINT  = "https://script.google.com/macros/s/AKfycbxdVYynFuPHaHg73kTFzL5R0TYXrPA8Tr0adMEsb0LKUS5vpDgysktWLPTTcLWrS7Jz/exec"     # the class board (already filled in: you do not need to change it)
TOKEN     = "f-BEqiAfHhAc2O-b"
ALLOWANCE = 0.06                # dollars you may spend (the whole class shares one small budget; re-running this cell does not reset it)

if os.environ.get("ARENA_MOCK"):
    PROVIDER = "mock"
if os.environ.get("ARENA_NO_BOARD"):                  # used by the instructor's offline tests
    ENDPOINT = TOKEN = ""
KEY = None
if PROVIDER != "mock":
    try:
        from google.colab import userdata
        KEY = userdata.get("OPENAI_API_KEY")
    except Exception:
        KEY = os.environ.get("OPENAI_API_KEY")
try:                                                  # the board address and password, set once as Colab secrets
    from google.colab import userdata
    ENDPOINT = ENDPOINT or userdata.get("BOARD_ENDPOINT")
    TOKEN = TOKEN or userdata.get("BOARD_TOKEN")
except Exception:
    ENDPOINT = ENDPOINT or os.environ.get("BOARD_ENDPOINT", "")
    TOKEN = TOKEN or os.environ.get("BOARD_TOKEN", "")
ENDPOINT, TOKEN = "".join(ENDPOINT.split()), "".join(TOKEN.split())   # remove hidden spaces and line breaks from pasting
arena.setup(NICK, provider=PROVIDER, api_key=KEY, endpoint=ENDPOINT or None,
            token=TOKEN or None, allowance_usd=ALLOWANCE)
arena.set_round("R3")

In [ ]:
CASE = "municipal"      # "municipal", "survey" or "cascade"   (your team draws one)
CASES = {
 "municipal": dict(task="B1", n_items=30000, budget=3.0,
                   goal="Maximise macro-F1 for 30,000 Turkish citizen complaints within $3."),
 "survey":    dict(task="B4", n_items=20000, budget=2.0,
                   goal="Maximise macro-F1 for 20,000 open-ended survey answers within $2, using ONE model."),
 "cascade":   dict(task="B4", n_items=20000, budget=2.0,
                   goal="Same survey data and the same $2, but a cheap model labels everything and a strong model only the items the cheap one is unsure about."),
}
C = CASES[CASE]
print(f"CASE: {CASE}\n{C['goal']}\nBudget ${C['budget']}  |  items {C['n_items']:,}")
print(f"=> maximum affordable cost per item: ${C['budget']/C['n_items']:.5f}  (per 1,000 items: ${C['budget']/C['n_items']*1000:.2f})")
arena.peek(C["task"], k=3)

## Step 1. Look at real inputs, outputs and tokens (before predicting anything)
This runs your prompt on **3 real items** and shows what goes in, what comes out, and what it costs. A *token* is a small piece of text (about 4 characters in English; more per word in Turkish).

In [ ]:
PROMPT = '''Classify the text.
Options: {labels}

Codebook:
{codebook}

Text: {text}'''
peek = arena.token_peek(PROMPT, task=C["task"], k=3, n_items=C["n_items"], budget=C["budget"])

**Before you go on, discuss in your team (1 minute):**
- How many tokens go **in** and how many come **out**? Which side is bigger?
- Where do the input tokens come from? (Your instructions and the codebook, or the item?)
- Which change would shrink the bill most: a shorter prompt, a cheaper model, or a shorter answer?

## Step 2. Predict the bill
Start from the numbers above and **adjust for the change you plan** (a shorter prompt lowers tokens in; asking for the label only lowers tokens out; a different model changes the price).

In [ ]:
PLAN_MODEL = "gpt-4o-mini"            # the model you plan to use for the whole job (allowed: gpt-5-nano, gpt-4o-mini, gpt-5.4-nano, gpt-5.4-mini)
TOKENS_IN, TOKENS_OUT = round(peek["tokens_in"]), round(peek["tokens_out"])      # start from what you measured, then edit
est = arena.estimate_cost(TOKENS_IN, TOKENS_OUT, C["n_items"], model=PLAN_MODEL)
print(f"Your prediction on {est['model']}: ${est['projected_usd']:.2f} for {est['n_items']:,} items  (${est['per_1k_items']:.3f} per 1,000)")
print("WITHIN BUDGET" if est["projected_usd"] <= C["budget"] else "OVER BUDGET: change the plan before you run anything")

## Step 3. Run a sample and compare with your prediction

In [ ]:
PRED_PER_1K = est["per_1k_items"]
res = arena.evaluate(PROMPT, task=C["task"], n=24, model=PLAN_MODEL, condition=CASE,
                     note=f"case={CASE}; model={PLAN_MODEL}; predicted_cost_per_1k={PRED_PER_1K:.3f}; H: baseline")
res.summary()
full = res.metrics["cost_per_1k"] * C["n_items"] / 1000
print(f"\nActual projected cost for the full job: ${full:.2f}  vs budget ${C['budget']}")
print(f"Estimation error: {abs(res.metrics['cost_per_1k']-PRED_PER_1K)/max(PRED_PER_1K,1e-9):.0%}")
print("WITHIN BUDGET" if full <= C["budget"] else "OVER BUDGET: pull a lever")

## Step 3b. The model ladder
The **same prompt** on three models. Compare quality, cost, and *how* they fail.
**Watch the cost column:** the model with the lowest *price per token* is not necessarily the cheapest *per item*. Some models spend hidden reasoning tokens. (The strongest model is not allowed in class; the instructor may demo it.)

In [ ]:
for mdl in ["gpt-5-nano", "gpt-4o-mini", "gpt-5.4-mini"]:
    r = arena.evaluate(PROMPT, task=C["task"], n=24, model=mdl, condition=CASE, note=f"case={CASE}; model ladder: {mdl}")
    print(f"{mdl:<14} F1 {r.metrics['macro_f1']:.2f}   cost per 1,000 items ${r.metrics['cost_per_1k']:.2f}\n")

## Step 4. Levers to try (one at a time, with a hypothesis)
| Lever | Try |
|---|---|
| **Model tier** | `model="..."` |
| **Prompt length** | Drop the codebook or shorten it (Step 1 showed where your tokens go) |
| **Prompt order for caching** | Put stable instructions and examples **first**, the changing item **last** (already how `{text}` works) |
| **Batching** | Several items per call: lower per-item overhead, but watch accuracy |
| **Output length** | Ask for the label only, no reasoning |
| **Cascade** | Cheap model first, strong model for low-confidence items (cell below) |

In [ ]:
# Example lever: a shorter prompt. Change one thing and write the hypothesis.
PROMPT_SHORT = "Label the text with one of: {labels}\n\nText: {text}"
res_b = arena.evaluate(PROMPT_SHORT, task=C["task"], n=24, model=PLAN_MODEL, condition=CASE,
                       note=f"case={CASE}; model={PLAN_MODEL}; H: dropping the codebook saves cost with a small F1 loss")
res_b.summary()

## Step 5. Cascade (the cascade case; everyone may try it)
A cheap model labels everything and reports a confidence. A strong model re-labels only the items below a threshold. Plot cost against quality for three thresholds and compare with your single-model result.

In [ ]:
CHEAP, STRONG = "gpt-4o-mini", "gpt-5.4-mini"      # allowed: gpt-5-nano, gpt-4o-mini, gpt-5.4-nano, gpt-5.4-mini
for th in (50, 70, 90):
    r = arena.cascade(PROMPT, task=C["task"], cheap=CHEAP, strong=STRONG, threshold=th, n=24, note=f"case={CASE}; threshold={th}")
    print(f"threshold {th}: F1 {r.metrics['macro_f1']:.3f}  cost/1k ${r.metrics['cost_per_1k']:.2f}\n")

### Team reflection
Which lever bought the most F1 per dollar? Was your prediction close, and what did you forget? What would you change if the budget were cut in half?